In [6]:
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizer, TFAutoModelForSequenceClassification, TFBertMainLayer
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# 1. 데이터 로드 및 전처리 (이전과 동일)
csv_file_path = 'labeled_reviews_40k.csv' 
df = pd.read_csv(csv_file_path)

df = df[['text', 'label']] 
df.dropna(subset=['text', 'label'], inplace=True)

label_mapping = {'부정': 0, '중립': 1, '긍정': 2}
df['label'] = df['label'].map(label_mapping)
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int) 

print(f"로드된 데이터 샘플:\n{df.head()}")
print(f"전체 데이터 수: {len(df)}")
print(f"라벨 분포:\n{df['label'].value_counts()}")

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"\n학습 데이터 수: {len(train_df)}")
print(f"검증 데이터 수: {len(val_df)}")

# 2. 토크나이저 및 TensorFlow BERT 모델 로드 (TFAutoModelForSequenceClassification은 바로 사용하지 않고, 그 내부의 BERT 레이어를 가져옵니다)
MODEL_NAME = 'klue/bert-base' 
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 사전 학습된 BERT의 핵심 부분 (pre-trained body)만 로드
# 여기서는 TFBertModel을 직접 사용하거나, TFAutoModelForSequenceClassification이 로드하는
# 내부의 BERT 레이어를 사용한다고 가정합니다.
# 직접적으로 TFBertModel을 사용하는 것이 더 명확합니다.
from transformers import TFBertModel
bert_model_core = TFBertModel.from_pretrained(MODEL_NAME)


# 3. Custom BERT Classification Model 정의
class CustomBertForSequenceClassification(tf.keras.Model):
    def __init__(self, bert_model_core, num_labels, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.bert = bert_model_core
        self.dropout = Dropout(dropout_rate)
        self.classifier = Dense(num_labels, name="classifier")

    def call(self, inputs, training=False):
        # inputs는 딕셔너리 형태의 텐서여야 합니다: {'input_ids': ..., 'attention_mask': ...}
        # token_type_ids가 필요하면 여기에 추가
        outputs = self.bert(inputs['input_ids'], attention_mask=inputs['attention_mask'], training=training)
        
        # BERT 출력의 [CLS] 토큰에 해당하는 마지막 은닉 상태를 가져옵니다.
        # pooler_output 또는 last_hidden_state[:, 0, :]를 사용할 수 있습니다.
        # sequence_output은 [batch_size, sequence_length, hidden_size] 형태
        # pooler_output은 [batch_size, hidden_size] 형태 (BERT의 CLS 토큰 처리 후 Dense 레이어를 거친 것)
        pooled_output = outputs.pooler_output # 또는 outputs.last_hidden_state[:, 0, :]
        
        pooled_output = self.dropout(pooled_output, training=training)
        logits = self.classifier(pooled_output)
        return logits

# 모델 인스턴스 생성
model = CustomBertForSequenceClassification(bert_model_core, num_labels=len(label_mapping))

# 모델의 입력 형태를 명시적으로 빌드 (이 부분이 중요합니다)
# Keras 모델은 .fit() 호출 전에 입력 shape를 알아야 합니다.
# dummy input으로 한 번 호출하여 모델의 내부 layer들을 build합니다.
dummy_inputs = {
    'input_ids': tf.zeros((1, MAX_LEN), dtype=tf.int32),
    'attention_mask': tf.zeros((1, MAX_LEN), dtype=tf.int32)
}
_ = model(dummy_inputs) # 모델을 한 번 호출하여 build
model.summary() # 모델 구조 확인

# 4. 데이터 토큰화 및 TensorFlow Dataset으로 변환 (이전과 동일)
MAX_LEN = 128 

def tokenize_data(df, tokenizer, max_len):
    input_ids = []
    attention_masks = []
    labels = df['label'].values

    for text in df['text']:
        encoded_dict = tokenizer.encode_plus(
                            str(text), 
                            add_special_tokens=True,
                            max_length=max_len,
                            padding='max_length',
                            return_attention_mask=True,
                            truncation=True,
                            return_tensors='tf', 
                       )
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])
    
    input_ids = tf.concat(input_ids, axis=0)
    attention_masks = tf.concat(attention_masks, axis=0)
    
    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks
    }, np.array(labels) 

X_train, y_train = tokenize_data(train_df, tokenizer, MAX_LEN)
X_val, y_val = tokenize_data(val_df, tokenizer, MAX_LEN)


# 5. 모델 컴파일 및 학습
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = ['accuracy']

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

EPOCHS = 5 
BATCH_SIZE = 16 

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1 
)

print("\n학습 완료!")

# 6. 모델 평가 (최종)
y_pred_logits = model.predict(X_val)
y_pred = np.argmax(y_pred_logits, axis=1) # 로짓에서 가장 높은 확률의 인덱스 선택

print("\n최종 분류 리포트:")
target_names = list(label_mapping.keys())
print(classification_report(y_val, y_pred, target_names=target_names))
print(f"최종 정확도: {accuracy_score(y_val, y_pred):.4f}")

# 7. 새로운 리뷰 예측 함수
def predict_sentiment_tf(text, model, tokenizer, max_len, label_mapping):
    encoded_input = tokenizer.encode_plus(
        str(text),
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        return_attention_mask=True,
        truncation=True,
        return_tensors='tf' 
    )

    # 모델 예측
    output_logits = model(encoded_input, training=False) # training=False로 설정하여 dropout 비활성화
    probabilities = tf.nn.softmax(output_logits, axis=-1).numpy()[0] 
    predicted_label_idx = np.argmax(probabilities)

    reverse_label_mapping = {v: k for k, v in label_mapping.items()}
    predicted_sentiment = reverse_label_mapping[predicted_label_idx]
    
    return predicted_sentiment, probabilities

test_review1 = "이 제품 정말 만족합니다. 가격 대비 성능 최고예요!"
test_review2 = "그냥 평범해요. 다시 구매할지는 모르겠습니다."
test_review3 = "최악의 경험. 절대 추천하지 않습니다."

print("\n새로운 리뷰 예측 결과:")
sentiment1, prob1 = predict_sentiment_tf(test_review1, model, tokenizer, MAX_LEN, label_mapping)
print(f"'{test_review1}' -> 예측 감성: {sentiment1}, 확률: {prob1}")

sentiment2, prob2 = predict_sentiment_tf(test_review2, model, tokenizer, MAX_LEN, label_mapping)
print(f"'{test_review2}' -> 예측 감성: {sentiment2}, 확률: {prob2}")

sentiment3, prob3 = predict_sentiment_tf(test_review3, model, tokenizer, MAX_LEN, label_mapping)
print(f"'{test_review3}' -> 예측 감성: {sentiment3}, 확률: {prob3}")

# 모델 저장 (선택 사항)
# 모델과 토크나이저를 저장할 디렉토리 생성
import os
save_directory = './my_custom_bert_sentiment_model'
if not os.path.exists(save_directory):
    os.makedirs(save_directory)

# Keras 형식으로 모델 저장 (weight만 저장하는 것을 권장)
model.save_weights(os.path.join(save_directory, 'tf_model.h5'))
# 토크나이저도 함께 저장
tokenizer.save_pretrained(save_directory)

# 모델을 다시 로드하는 방법 (나중에 사용 시)
# loaded_bert_model_core = TFBertModel.from_pretrained(MODEL_NAME)
# loaded_model = CustomBertForSequenceClassification(loaded_bert_model_core, num_labels=len(label_mapping))
# loaded_model(dummy_inputs) # 모델 구조 빌드
# loaded_model.load_weights(os.path.join(save_directory, 'tf_model.h5'))

로드된 데이터 샘플:
                                                text  label
0  댓글 보면 아시겠지만 구매한 사람은 높은 별점을 준 반면에, 사서 읽어보지도 않은 ...      1
1     "나는 빠리의 택시운전사"보다 조금 자세히 기술한 문화 차이 그리고 한국사회 문제점      1
2                 왜, 우린, 모든 것의 기본인 '대화'조가 시도할 수 없는가?      1
3                                        좋네요~한번읽어볼듯~      2
4              잔글씨로 빼곡히 적어진 내용이 빌게이츠의 생각을 그대로 옮겨놓았네요      1
전체 데이터 수: 304004
라벨 분포:
label
2    198556
1     69524
0     35924
Name: count, dtype: int64

학습 데이터 수: 243203
검증 데이터 수: 60801


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already u

Model: "custom_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 tf_bert_model (TFBertModel)  multiple                 110617344 
                                                                 
 dropout_75 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  2307      
                                                                 
Total params: 110,619,651
Trainable params: 110,619,651
Non-trainable params: 0
_________________________________________________________________
Epoch 1/5
15201/15201 [==============================] - 3194s 209ms/step - loss: 0.3460 - accuracy: 0.8553 - val_loss: 0.3153 - val_accuracy: 0.8656
Epoch 2/5
15201/15201 [==============================] - 3103s 204ms/step - loss: 0.2221 - accuracy: 0.9100 - val_loss: 0.3125 

('./my_custom_bert_sentiment_model\\tokenizer_config.json',
 './my_custom_bert_sentiment_model\\special_tokens_map.json',
 './my_custom_bert_sentiment_model\\vocab.txt',
 './my_custom_bert_sentiment_model\\added_tokens.json')

In [3]:
!pip install transformers

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ---------------------------------------- 10.5/10.5 MB 54.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 46.2 MB/s eta 0:00:00
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ---------- ----------------------------- 2/8 [regex]
   --------------- ------------------------ 3/8 [fsspec]
   ------------------------- -------------- 5/8 [huggingface-hub]
   ------------------------- -------------- 5/8 [huggingface-hub]
   ----------------------------------- ---- 7/8 [transformers]
   ----------------------------------- ---- 7/8 [transformers]
   ----------------------------------- ---- 7/8 [transformers]
   ----------------------------------- ---- 7/8 [transformers]
   ----------------------------------- ---- 7/8 [transformers]
   ------------------------------